# Geometric Priors I

This notebook follows **Chapter 3, "Geometric Priors"**, of

> M. M. Bronstein, J. Bruna, T. Cohen, and P. Veličković,  
> *Geometric Deep Learning: Grids, Groups, Graphs, Geodesics, and Gauges*.

The scope here is **the beginning of Chapter 3 through the end of §3.2** (printed pages 9–19).

The chapter is deliberately concise, so this notebook expands the mathematical ideas enough to make the geometric viewpoint comfortable.

We will **not repeat material developed in earlier notebooks**:

- Groups, subgroups, group actions, and basic group representations → see `02_groups.ipynb`.
- Vector spaces, tensor spaces, norms, metrics, and inner products → see `03_linear_spaces.ipynb`.
- The curse of dimensionality and generic inductive bias → see `04_learning_in_high_dimension.ipynb`.

Instead, we will concentrate on the new ideas introduced here:

1. geometric priors as a response to high-dimensional learning;
2. the three spaces: domain, signals, and hypothesis class;
3. how domain symmetries induce actions on signals;
4. why the representation on a feature space need not look like the original domain action;
5. invariance versus equivariance as architectural constraints;
6. the hierarchy of structures and its relation to subgroups;
7. isomorphisms versus automorphisms;
8. graph isomorphism and automorphism as a preview of later chapters.

## 1. From the curse of dimensionality to geometric priors

Chapter 2 showed that generic high-dimensional function classes are difficult to learn efficiently.

The important observation at the beginning of Chapter 3 is that **real-world data are rarely generic**. They usually arise from an underlying physical or mathematical domain with additional structure.

The proto-book proposes two broad sources of useful structure:

$$
\boxed{\text{symmetry}}
\qquad\text{and}\qquad
\boxed{\text{scale separation}}.
$$

These are called **geometric priors** because they encode assumptions about how the data are organised geometrically.

The philosophy is:

$$
\text{domain structure}
\longrightarrow
\text{restricted admissible transformations}
\longrightarrow
\text{restricted hypothesis class}.
$$

Instead of asking a neural network to learn an arbitrary function on a huge space, we build the transformations and regularities into the model that the problem already suggests.

### Why this is a different kind of inductive bias

In `04_learning_in_high_dimension.ipynb`, we saw examples where one restricts a function class by a generic complexity measure, such as a norm or a smoothness assumption.

A geometric prior is different in character.

For example, suppose an image classification problem has the property that translating an object does not change its class.

Then instead of merely saying that the classifier should be "simple", we can impose the much more specific condition

$$
f(\rho(g)x)=f(x),
$$

where $g$ represents an allowed transformation such as a translation.

This eliminates many possible hypotheses **because they violate a known symmetry of the task**.

The restriction is therefore not arbitrary regularisation. It is a statement about the geometry of the problem.

## 2. Signals live on domains

The central abstraction introduced by the proto-book is that data should be regarded as **signals on a domain**.

Let

$$
\Omega
$$

be a domain and let

$$
\mathcal C
$$

be a vector space of channels. A signal is a function

$$
x:\Omega\to\mathcal C.
$$

The collection of such signals is

$$
\mathcal X(\Omega,\mathcal C)
=
\{x:\Omega\to\mathcal C\}.
$$

We usually suppress $\mathcal C$ and write simply $\mathcal X(\Omega)$.

This viewpoint is more general than thinking of data as vectors in $\mathbb R^d$.

For example:

- an image is a signal on a pixel grid;
- a time series is a signal on a temporal domain;
- a graph signal is a signal on the vertices of a graph;
- a function on a manifold is a signal on that manifold. (People who knows Quantum Field Theory should recognise this as a field.)

The important point is that the coordinates of the data acquire meaning from **where they live on the domain**.

### Channels versus domain dimension

The proto-book emphasises that the codomain $\mathcal C$ represents **channels**.

For an RGB image,

$$
\Omega=\mathbb Z_n\times\mathbb Z_n,
\qquad
\mathcal C=\mathbb R^3.
$$

Thus

$$
x:\Omega\to\mathbb R^3.
$$

The spatial domain tells us *where* the signal is sampled; the channels tell us *what values are attached to each location*.

This distinction becomes extremely important later.

A transformation of the domain may move locations without changing channels, while a learned representation may transform its feature channels in a nontrivial way.

That is why the group acting on the domain and the representation carried by a neural-network feature space should not be identified too quickly.

## 3. The signal space and its geometry

We already developed the vector-space and inner-product structure of signal spaces in `03_linear_spaces.ipynb`.

The proto-book writes, for signals $x,y\in\mathcal X(\Omega,\mathcal C)$,

$$
(\alpha x+\beta y)(u)
=
\alpha x(u)+\beta y(u).
$$

If $\mathcal C$ has an inner product and $\Omega$ has a measure $\mu$, then

$$
\langle x,y\rangle
=
\int_\Omega
\langle x(u),y(u)\rangle_{\mathcal C}
\,d\mu(u).
$$

(**Sorry!** The concept of measure has not been introduced, but the protobook doesn't use it in full strength. I will try add a notebook on measure theory towards the end.)

For a discrete domain, the integral becomes a sum.

We will not develop this again here. See:

> `03_linear_spaces.ipynb`, especially the sections on tensor spaces, norms, metrics, and inner products.

What is new here is what happens when **transformations of $\Omega$ act on this signal space**.


## 4. Scale separation

The proto-book identifies **scale separation** as the second major geometric prior alongside symmetry.

The basic idea is different from symmetry.

A symmetry relates descriptions of the **same object at the same scale**:

$$
x \longmapsto \rho(g)x.
$$

Scale separation instead relates descriptions at **different resolutions**.

Suppose

$$
\Omega_{\mathrm{fine}}
\longrightarrow
\Omega_{\mathrm{coarse}}
$$

is a coarsening of the domain. We would like a corresponding map between signal spaces,

$$
C:
\mathcal X(\Omega_{\mathrm{fine}})
\longrightarrow
\mathcal X(\Omega_{\mathrm{coarse}}),
$$

that retains the information relevant to the task.

The important phrase is **relevant information**. Coarsening is generally lossy: it cannot preserve every detail of the original signal.


### What does it mean for information to survive coarsening?

Consider a one-dimensional signal sampled on a fine grid,

$$
x_0,x_1,\ldots,x_{N-1}.
$$

A simple coarsening might replace every pair of neighbouring values by their average:

$$
(Cx)_k
=
\frac{x_{2k}+x_{2k+1}}{2}.
$$

Rapid oscillations can disappear under this operation, while slowly varying structure can survive.

For example, a low-frequency signal

$$
x_j=\sin\left(\frac{2\pi j}{N}\right)
$$

changes slowly from one neighbouring point to the next, whereas a high-frequency signal can oscillate strongly inside each pair.

Thus a useful hierarchy of scales can separate:

$$
\text{fine details}
\qquad\text{from}\qquad
\text{coarser structure}.
$$

This is the conceptual reason pooling, subsampling, multiresolution representations, and related operations can be useful.

We will study their concrete implementations later rather than introducing a CNN here.


### Symmetry and scale separation are complementary

It is useful to keep the two geometric priors conceptually distinct.

**Symmetry** asks:

$$
\text{How should the representation change when the domain is transformed?}
$$

**Scale separation** asks:

$$
\text{Which information survives when the domain is represented more coarsely?}
$$

A modern architecture can exploit both simultaneously.

For an image, for example:

$$
\text{translation symmetry}
\quad+\quad
\text{multiscale spatial structure}.
$$

The first motivates weight sharing and equivariance; the second motivates hierarchical or coarse-grained representations.

This distinction will become important when we later study the blueprint of GDL architectures.

## 4. The three spaces of Geometric Deep Learning

One of the most useful conceptual pictures in this chapter is the distinction between three spaces:

$$
\boxed{
\Omega
\quad\longrightarrow\quad
\mathcal X(\Omega)
\quad\longrightarrow\quad
\mathcal F(\mathcal X(\Omega))
}
$$

where:

- $\Omega$ is the **physical or geometric domain**;
- $\mathcal X(\Omega)$ is the **space of signals** on that domain;
- $\mathcal F(\mathcal X(\Omega))$ is the **hypothesis class**, consisting of functions acting on signals.

A symmetry group $G$ first acts on the domain $\Omega$.

That action induces an action on signals.

The induced action then constrains the admissible functions in the hypothesis class.

Schematically,

$$
G\curvearrowright\Omega
\quad\Longrightarrow\quad
G\curvearrowright\mathcal X(\Omega)
\quad\Longrightarrow\quad
\text{constraints on }f\in\mathcal F(\mathcal X(\Omega)).
$$

This is the central bridge from geometry to machine learning.

### Why the three-space picture matters

It prevents a common conceptual confusion.

A transformation can act at different levels:

1. **Domain level:** move points of $\Omega$.
2. **Signal level:** move the values of a signal along with the domain.
3. **Feature level:** transform an internal representation produced by a neural network.
4. **Output level:** possibly remove the transformation entirely through an invariant readout.

The same abstract group $G$ can therefore appear several times in a network, with different representations at different stages.

This is one reason representation theory becomes so useful in GDL.

## 5. From domain actions to signal actions

We covered group actions in `02_groups.ipynb`. Here the important new point is how a domain action automatically produces an action on signals.

Suppose

$$
G\curvearrowright\Omega,
\qquad
u\mapsto g.u.
$$

Then the induced action on a signal is

$$
(g.x)(u)
=
x(g^{-1}.u).
$$

The inverse is essential.

It ensures that the induced transformation respects composition:

$$
g.(h.x)=(gh).x.
$$

### Why the inverse appears

Suppose a signal contains a pattern located at $u$.

If the domain is transformed by $g$, the pattern should move to $g.u$.

To determine the new value at a location $u$, we therefore ask:

> Where did the value currently sitting at $u$ come from?

It came from the old location $g^{-1}.u$.

Hence

$$
(g.x)(u)=x(g^{-1}.u).
$$

This is the same pullback construction that appears naturally when functions are transformed by changes of coordinates.

### A useful mathematical interpretation

The domain action

$$
g:\Omega\to\Omega
$$

acts on functions by **precomposition with $g^{-1}$**:

$$
x\mapsto x\circ g^{-1}.
$$

Thus the representation on the signal space is not an arbitrary extra definition. It is canonically induced by the action on the domain.

This idea will recur throughout GDL:

$$
\text{transformation of the domain}
\quad\Rightarrow\quad
\text{transformation of functions on the domain}.
$$

## 6. Representations: the same group can act in different ways

A representation is a homomorphism

$$
\rho:G\to GL(V),
$$

so that

$$
\rho(gh)=\rho(g)\rho(h).
$$

We introduced representations already in `02_groups.ipynb`. The point emphasized here is subtler:

> **The dimension of the representation space is not determined by the dimension of the group or the domain.**

For example, a two-dimensional translation group can act on:

- an image with millions of pixels;
- a small feature vector;
- an intermediate feature map;
- a scalar output.

The underlying group can be the same while the representation spaces are completely different.

This is exactly what makes representation theory useful for neural networks: we can ask how the symmetry should act on each learned feature space.

### Feature representations are learned objects

Suppose a group $G$ acts on an input space through

$$
\rho_{\mathrm{in}}(g).
$$

A neural network layer may map this space to another representation space:

$$
\Phi:
V_{\mathrm{in}}\to V_{\mathrm{out}}.
$$

If the layer is equivariant, then

$$
\Phi\bigl(\rho_{\mathrm{in}}(g)x\bigr)
=
\rho_{\mathrm{out}}(g)\Phi(x).
$$

Notice that the two representations need not be identical:

$$
\rho_{\mathrm{in}}
\neq
\rho_{\mathrm{out}}.
$$

The network is therefore not simply "preserving the input". It is transporting the symmetry information through a sequence of representation spaces.

## 7. Equivariant maps as intertwiners

There is a useful representation-theoretic term for an equivariant map.

If

$$
\Phi:V\to W
$$

satisfies

$$
\Phi(\rho_V(g)v)
=
\rho_W(g)\Phi(v)
$$

for every $g\in G$, then $\Phi$ is called an **intertwining map**, or simply an **intertwiner**, between the two representations.

Equivalently, for every $g$,

$$
\boxed{
\Phi\circ\rho_V(g)
=
\rho_W(g)\circ\Phi.
}
$$

This gives a commutative diagram:

$$
\begin{array}{ccc}
V & \xrightarrow{\ \Phi\ } & W\\
\downarrow{\rho_V(g)} && \downarrow{\rho_W(g)}\\
V & \xrightarrow{\ \Phi\ } & W
\end{array}
$$

The two paths from the upper-left corner to the lower-right corner agree.

This language will become increasingly useful when we study equivariant neural-network layers.

### Why this is stronger than "data augmentation"

Data augmentation exposes a model to transformed examples.

Equivariance instead constrains the **function represented by the architecture**:

$$
\Phi(\rho(g)x)=\rho'(g)\Phi(x).
$$

Thus the transformation rule is part of the architecture's hypothesis space.

This is a structural distinction:

- augmentation says, roughly, *"please learn from these transformed examples"*;
- equivariance says, *"the transformation must propagate through this map in this prescribed way"*.

They can be used together, but they are not mathematically the same mechanism.

## 8. Invariance and equivariance serve different tasks

We already defined these notions in `02_groups.ipynb`; here we focus on how they fit together.

### Invariant output

For

$$
f:\mathcal X(\Omega)\to Y,
$$

invariance means

$$
f(\rho(g)x)=f(x).
$$

The transformation disappears at the output.

This is appropriate when the target does not depend on the transformation.

Examples include:

- object category under an allowed translation;
- a molecular property independent of overall orientation;
- a graph-level property independent of node ordering.

### Equivariant output

For

$$
F:\mathcal X(\Omega)\to\mathcal X(\Omega'),
$$

equivariance means

$$
F(\rho(g)x)=\rho'(g)F(x).
$$

The transformation remains visible in the output.

This is appropriate when the output itself has geometric meaning.

For example, a segmentation mask should move when the input image moves.

### The architecture pattern

The proto-book highlights the pattern

$$
\boxed{
\text{equivariant processing}
\quad\longrightarrow\quad
\text{invariant readout}
}
$$

for tasks whose final answer should be invariant.

Schematically,

$$
x
\xrightarrow{\text{equivariant}}
h_1
\xrightarrow{\text{equivariant}}
h_2
\xrightarrow{\text{equivariant}}
h_L
\xrightarrow{\text{invariant}}
y.
$$

The internal representations retain information about *where or how* the input has transformed, while the final readout discards that information when it is irrelevant to the task.

This pattern is broader than CNNs. The proto-book later points out that the same blueprint appears in graph neural networks and other geometric architectures.

## 9. Symmetry is task-dependent

A crucial point in the chapter is that **there is no universally correct symmetry group**.

A transformation is a symmetry only relative to a particular task.

Suppose $G_1$ is a larger group of transformations and $G_2\subseteq G_1$ is a smaller subgroup.

If the target is invariant under $G_1$, then imposing $G_2$-invariance is weaker.

But if the target is only invariant under $G_2$, imposing invariance under all of $G_1$ can destroy information that is relevant to the task.

The proto-book gives a concrete example:

- for some histopathology tasks, reflections may be acceptable symmetries;
- for road-sign classification, reflections may change the meaning of the sign.

Thus geometric prior selection is itself a modeling decision.

## 10. Levels of structure

Section 3.2 introduces an important refinement.

A domain can carry **different amounts of structure**.

Start with a bare set:

$$
\Omega.
$$

At this level, the only structure is its collection of elements and its cardinality.

Then we can add more structure:

$$
\begin{array}{c}
\text{set}\\
\downarrow\\
\text{topological space}\\
\downarrow\\
\text{smooth manifold}\\
\downarrow\\
\text{metric space}\\
\downarrow\\
\text{oriented geometric space}
\end{array}
$$

Each additional structure restricts which transformations count as equivalences.

### Set structure

For a set, the structure-preserving invertible maps are simply bijections

$$
\tau:\Omega\to\Omega.
$$

For a finite set with $n$ elements, these form the symmetric group

$$
S_n.
$$

Thus, with no additional structure, **every relabelling is a symmetry**.

This is a very weak notion of geometry.

If we later put additional structure on $\Omega$, most of these permutations will cease to preserve it.

### Topological structure

If $\Omega$ is a topological space, a structure-preserving bijection is a **homeomorphism**:

$$
\tau:\Omega\to\Omega'
$$

such that both

$$
\tau
\quad\text{and}\quad
\tau^{-1}
$$

are continuous.

The inverse-continuity condition matters.

A continuous bijection need not be a homeomorphism in general.

The geometric intuition is that a homeomorphism preserves the qualitative notion of neighbourhood and continuity, while allowing stretching and bending.

### Smooth structure

For smooth manifolds, the corresponding structure-preserving maps are **diffeomorphisms**.

A diffeomorphism is a bijection

$$
\tau:\Omega\to\Omega'
$$

such that both $\tau$ and $\tau^{-1}$ are smooth.

Thus we are now preserving differentiable structure, not merely topology.

This distinction will become important later when the proto-book moves from discrete domains to manifolds.

### Metric structure

If $\Omega$ carries a metric $d$, an isomorphism preserving the metric is an **isometry**:

$$
d(\tau(u),\tau(v))
=
d(u,v).
$$

We already developed metric spaces in `03_linear_spaces.ipynb`; the new point here is their role in determining which transformations qualify as symmetries.

The same underlying set can therefore have very different symmetry groups depending on which metric or other structure we choose.

## 11. The hierarchy of symmetry groups

Suppose a set carries increasingly rich structures.

Then the transformations preserving the richer structure form smaller groups.

A schematic example is

$$
\mathrm{SE}(2)
\subseteq
\mathrm E(2)
\subseteq
\mathrm{Diff}(\mathbb R^2).
$$

Here:

- $\mathrm{Diff}(\mathbb R^2)$ contains all diffeomorphisms;
- $\mathrm E(2)$ contains Euclidean isometries;
- $\mathrm{SE}(2)$ contains orientation-preserving Euclidean isometries.

Thus

$$
\text{more structure preserved}
\quad\Longrightarrow\quad
\text{fewer admissible symmetries}.
$$

This is one of the clearest ways to connect the Erlangen viewpoint to GDL:

$$
\boxed{
\text{geometry}
\;\longleftrightarrow\;
\text{invariants}
\;\longleftrightarrow\;
\text{symmetry group}.
}
$$

### Why the subgroup relation is important for GDL

Suppose

$$
H\subseteq G.
$$

If a function is $G$-invariant,

$$
f(\rho(g)x)=f(x)
\qquad\forall g\in G,
$$

then it is automatically $H$-invariant because every element of $H$ is also an element of $G$.

But the converse need not hold.

Therefore moving to a **smaller symmetry group** generally allows a larger hypothesis class.

This gives a useful mental picture:

$$
\text{larger symmetry group}
\Rightarrow
\text{stronger constraint}
\Rightarrow
\text{smaller admissible function class}.
$$

*The choice of geometric structure therefore directly affects the inductive bias of the learning system.*

## 12. Isomorphisms versus automorphisms

The distinction is simple but fundamental.

An **isomorphism** is a structure-preserving invertible map between two potentially different objects:

$$
\eta:\Omega\to\Omega'.
$$

An **automorphism** is an isomorphism from an object to itself:

$$
\tau:\Omega\to\Omega.
$$

Thus

$$
\boxed{
\text{automorphism}
=
\text{self-isomorphism}.
}
$$

Automorphisms are the mathematical version of the word **symmetry** used throughout the chapter.

### A simple set example

Let

$$
\Omega=\{0,1,2\},
\qquad
\Omega'=\{a,b,c\}.
$$

The map

$$
\eta(0)=a,\qquad
\eta(1)=b,\qquad
\eta(2)=c
$$

is a set isomorphism.

It compares two different sets with the same cardinality.

On the other hand,

$$
\tau(u)=u+1\pmod 3
$$

is an automorphism of $\Omega$.

It compares $\Omega$ with itself and therefore represents a symmetry of that object.

### Why the distinction matters in machine learning

Suppose two domains are isomorphic.

If a learning problem depends only on the structure of the domain and not on arbitrary labels, then an algorithm should ideally give corresponding answers on the two representations.

This is the deeper reason that isomorphisms matter for geometric learning:

> **The learned computation should depend on structure, not on arbitrary choices used to represent that structure.**

For graphs, this becomes especially important because node labels usually have no intrinsic meaning.

## 13. Graph isomorphism: the first important discrete example

We will study graphs systematically later, so this is only a preview.

For graphs

$$
G=(V,E),
\qquad
G'=(V',E'),
$$

an isomorphism is a bijection

$$
\eta:V\to V'
$$

such that adjacency is preserved:

$$
(u,v)\in E
\quad\Longleftrightarrow\quad
(\eta(u),\eta(v))\in E'.
$$

Two isomorphic graphs therefore have the same connectivity structure even if their vertices are labelled differently.

A graph automorphism is the special case

$$
\eta:V\to V.
$$

So a graph's automorphism group is precisely its symmetry group.

In [1]:
# A minimal computational demonstration of graph isomorphism.
# We use NetworkX only to verify the structural statement.
# The graph-theoretic machinery itself will be developed later.

import networkx as nx

# Construct a path on three vertices.
G = nx.path_graph(3)

# Relabel the vertices without changing the graph structure.
H = nx.relabel_nodes(G, {0: 2, 1: 0, 2: 1})

# NetworkX checks whether there is a structure-preserving bijection.
isomorphic = nx.is_isomorphic(G, H)

print("G and H are isomorphic:", isomorphic)
print("Edges of G:", sorted(G.edges()))
print("Edges of H:", sorted(H.edges()))

G and H are isomorphic: True
Edges of G: [(0, 1), (1, 2)]
Edges of H: [(0, 1), (2, 0)]


The point of this small experiment is not to learn NetworkX.

It demonstrates one conceptual distinction:

$$
\text{same labelled representation}
\neq
\text{same underlying object}.
$$

The two graphs have different vertex labels, but an appropriate bijection preserves their connectivity.

Later, when we study graph neural networks, permutation symmetry will make this issue central.

## 14. Automorphisms as actual symmetries

A graph can have many automorphisms.

The proto-book mentions the **Folkman graph** as an example with 3840 automorphisms. This is a useful reminder that symmetry is not necessarily visually obvious from one particular drawing.

An automorphism is a permutation of vertices that preserves adjacency:

$$
\tau:V\to V,
\qquad
(u,v)\in E
\Longleftrightarrow
(\tau(u),\tau(v))\in E.
$$

The collection

$$
\operatorname{Aut}(G)
$$

forms a group under composition.

The important GDL lesson is that **node identities are not necessarily geometric information**. A graph representation may assign arbitrary labels to vertices, while its automorphism group records which relabellings leave the actual structure unchanged.

### The Folkman graph

The Folkman graph was introduced by Jon Folkman in 1967. It has 20 vertices and 40 edges and is a classical example in the study of graph symmetry.

The proto-book uses it to illustrate that a graph can possess a very large automorphism group:

$$
|\operatorname{Aut}(G)|=3840.
$$

The exact number is less important here than the conceptual point:

> A symmetry group can encode a large collection of transformations that leave the mathematical object unchanged.

Later GDL constructions will exploit precisely this kind of structure rather than treating every ordering of the data as meaningful.

## 15. What has been established so far

The chapter has now moved us from generic learning to a geometric point of view.

The chain is:

$$
\boxed{
\text{high-dimensional data}
\rightarrow
\text{underlying domain}
\rightarrow
\text{domain structure}
\rightarrow
\text{symmetries}
\rightarrow
\text{restricted hypothesis classes}.
}
$$

More concretely:

1. Data are treated as **signals on domains**.
2. Domain symmetries act on signals.
3. Those actions are represented on feature spaces.
4. Neural-network maps can be required to be equivariant.
5. Final outputs can be made invariant when appropriate.
6. The choice of structure determines the admissible symmetry group.
7. Isomorphisms express equivalence between different representations.
8. Automorphisms express symmetries of a single object.

The next part of the chapter will relax the idealised assumption that symmetries are exact.

## 16. A note related to CNN

The proto-book describes CNNs as a canonical example in which:

- shared convolutional weights exploit translational symmetry;
- pooling exploits scale separation.

We will **not implement a CNN here**. A proper treatment requires its own notebook.

The important conceptual point for this chapter is instead the division of labour:

$$
\text{local equivariant operations}
\quad+\quad
\text{coarsening/readout}
$$

can turn geometric structure into an inductive bias.

This should be kept in mind when we later study the concrete architectures. The CNN is an example of the principle, not the definition of the principle.

## 17. Reference note: low-dimensional projections

The proto-book refers to Bach (2017) when discussing the possibility of breaking the curse of dimensionality through functions of low-dimensional projections,

$$
f(x)\approx g(Ax),
\qquad
A\in\mathbb R^{k\times d},
\quad k\ll d.
$$

Bach's work studies convex neural-network formulations with infinitely many hidden units and shows how appropriate regularisation can adapt to unknown low-dimensional linear structure. It also discusses sparsity-inducing regularisation for high-dimensional variable selection.

The significance for GDL is not that low-dimensional projections are useless. Rather:

> They provide one possible source of structure, but many physical signals have richer spatial organisation that is better described through the geometry of the domain.

This motivates looking for geometric priors directly on $\Omega$.

## 18. One conceptual picture

The first three notebooks supplied much of the algebraic and analytic language needed here.

We can now combine them:

$$
\begin{array}{c}
\text{domain }\Omega\\[2mm]
\downarrow\;G\text{-action}\\[2mm]
\text{signals }\mathcal X(\Omega)\\[2mm]
\downarrow\;\rho(g)\\[2mm]
\text{feature representations}\\[2mm]
\downarrow\;\text{equivariant maps}\\[2mm]
\text{task output}\\[2mm]
\downarrow\\[2mm]
\text{invariant quantity, when appropriate.}
\end{array}
$$

This is the conceptual machinery that will allow us to treat images, sets, graphs, manifolds, and other domains within one framework.

## Summary

The central lesson of the beginning of Chapter 3 is:

$$
\boxed{
\text{geometry provides structured inductive bias.}
}
$$

A domain $\Omega$ carries structure. Its symmetries form a group $G$. That group acts on signals and on learned representations. Requiring neural-network maps to respect these actions gives equivariance, while discarding the action at the output gives invariance.

The second major idea is that **the notion of symmetry depends on the structure we choose to preserve**:

$$
\text{set}
\rightarrow
\text{topology}
\rightarrow
\text{smoothness}
\rightarrow
\text{metric}
\rightarrow
\cdots
$$

and the corresponding symmetry groups become progressively smaller.

Finally,

$$
\text{isomorphism}
=
\text{structure-preserving equivalence},
$$

while

$$
\text{automorphism}
=
\text{structure-preserving symmetry}.
$$

These ideas will provide the foundation for the next parts of Geometric Deep Learning, where exact symmetry is relaxed to **deformation stability** and then combined with **scale separation**.

## Exercises

### Exercise 1 — Structure and symmetry

Take a finite set with four elements. Determine its symmetry group when:

1. only the set structure is specified;
2. a cycle graph structure is added;
3. a linear ordering is added.

How does adding structure change the allowed transformations?

### Exercise 2 — Representation spaces

Give three different vector spaces on which the same abstract group could act. Explain why the representation dimension need not equal the dimension of the group.

### Exercise 3 — Invariance versus equivariance

For each task, decide whether the desired output is naturally invariant or equivariant:

1. image classification;
2. semantic segmentation;
3. graph-level molecular property prediction;
4. node classification.

Justify each answer mathematically.

### Exercise 4 — Isomorphism versus automorphism

Give an example of:

- an isomorphism between two different mathematical objects;
- a non-trivial automorphism of one object.

### Exercise 5 — Levels of structure

Consider the plane $\mathbb R^2$. Explain why

$$
\mathrm{SE}(2)\subseteq \mathrm E(2)\subseteq\mathrm{Diff}(\mathbb R^2).
$$

What additional structure is being preserved at each stage?

### Exercise 6 — Intertwiners

Suppose

$$
F\rho_1(g)=\rho_2(g)F.
$$

Explain in words what this says about applying a symmetry before versus after $F$.

## References

### Main source

1. M. M. Bronstein, J. Bruna, T. Cohen, and P. Veličković,  
   **Geometric Deep Learning: Grids, Groups, Graphs, Geodesics, and Gauges**, arXiv:2104.13478, 2021.  
   Chapter 3, especially §§3–3.2.

### References explicitly cited in the selected Chapter 3 pages

2. F. Bach, **Breaking the Curse of Dimensionality with Convex Neural Networks**, arXiv:1412.8690, 2017.  
   The paper studies convex neural-network formulations and how regularisation can exploit low-dimensional linear structure.

3. J. Folkman, **Regular line-symmetric graphs**, *Journal of Combinatorial Theory* 3 (1967), 215–232.  
   The proto-book cites this work in connection with the Folkman graph and its large automorphism group.

### Previous notebooks

4. `02_groups.ipynb` — groups, group actions, representations, invariance, and equivariance.

5. `03_linear_spaces.ipynb` — vector spaces, tensor spaces, norms, metrics, and inner products.

6. `04_learning_in_high_dimension.ipynb` — inductive bias and the curse of dimensionality.

## Further reading

- U. von Luxburg and O. Bousquet, **Distance-Based Classification with Lipschitz Functions**, *Journal of Machine Learning Research* 5 (2004), 669–695. Their work connects Lipschitz function classes with metric geometry, margins, and maximum-discrepancy ideas.

- A. B. Tsybakov, **Introduction to Nonparametric Estimation**, Springer, 2009. This provides a deeper treatment of minimax rates and nonparametric estimation, including Sobolev classes.

- B. Neyshabur, R. Salakhutdinov, and N. Srebro, **Path-SGD: Path-Normalized Optimization in Deep Neural Networks**, arXiv:1506.02617, 2015.

- S. Gunasekar, B. Woodworth, S. Bhojanapalli, B. Neyshabur, and N. Srebro, **Implicit Regularization in Matrix Factorization**, arXiv:1705.09280, 2017.

- N. Razin and N. Cohen, **Implicit Regularization in Deep Learning May Not Be Explainable by Norms**, NeurIPS 2020.